# 01. Objects, names, references

**Tier:** Deep. **Why you'd use this:** Two variables seem to hold the same list, you change one, and the other changes too. You need a clear picture of names pointing at objects, and when to copy.

**After this lesson you can:** tell when two names share an object; tell `is` from `==`; choose a copy when a second name must not see a mutation.
About 30 minutes. **Uses:** [Names](../../course/01-basics/01-names-lesson.ipynb), [Lists](../../course/04-collections/01-lists-lesson.ipynb)


## Quick review

1. What does a name hold — a value's bytes, or a reference to an object?
2. What does `a = b` do for two names and one list?

<details><summary>Answers</summary>

A reference to an object. Both names point at the same list object.
</details>


## The idea

A value in Python is an object: a piece of memory with a type and an identity. A name is not the object. A name is an entry in a namespace that points at an object. Assignment writes that entry. It does not copy the object.

`id(obj)` is the identity, a number unique to that object while it is alive. `type(obj)` is the object's type. `a is b` is true when `a` and `b` point at the same object, which is the same fact as `id(a) == id(b)`. `a == b` asks the objects whether they count as equal. Two equal lists can still be two objects.

<details><summary>Explain more</summary>

The namespace for a function is its frame. The namespace for the notebook is the global one. Either way, the name is a label. Stick a second label on the same box and both labels see the same contents. Peel a label off and stick it on a new box, and the other label still sees the old box.

Paste each short example into [Python Tutor](https://pythontutor.com/visualize.html) and step it. The arrows there are the references this lesson is about.
</details>

<details><summary>If you already write Java, JavaScript, or C#</summary>

This is reference semantics for every value, including small numbers. There is no separate "the int is stored in the variable" case you can rely on. Rebinding a name is not a mutation of the object.
</details>


## See it work

A reading buffer is a list. A function records a snapshot name and then the caller appends a new sample. The snapshot must keep the old samples. Assignment is the wrong tool. A copy is a new list object.


In [1]:
def snapshot(samples: list[int]) -> list[int]:
    # 1. A new list object, holding the same numbers.
    frozen = list(samples)
    return frozen


samples = [10, 11]
frozen = snapshot(samples)
samples.append(12)
print(frozen)
print(samples)
print(frozen is samples)


[10, 11]
[10, 11, 12]
False


| step | samples | frozen | same object? |
| --- | --- | --- | --- |
| after `samples = [10, 11]` | [10, 11] | not bound yet | |
| after `frozen = list(samples)` | [10, 11] | [10, 11] | no |
| after `samples.append(12)` | [10, 11, 12] | [10, 11] | no |

```
samples ──► [10, 11, 12]
frozen  ──► [10, 11]
```

`list(samples)` walks the original and builds a new list. That costs time and memory proportional to the length. The assignment `frozen = samples` would have been a single pointer write, and the append would have shown up in both names. Step the copy version in [Python Tutor](https://pythontutor.com/visualize.html) and you will see two arrows to two lists.


## Check yourself

Answer these from the example above before you run the cell.


1. After `right = left` and `right.append(3)`, why does `left` change?
2. Why can `left == right` be true when `left is right` is false?
3. What does rebinding `right` change that `append` does not?

<details><summary>Answers</summary>

1. There is one list. `append` mutates that object. `left` still holds the pointer that `right` holds, so it sees the new element.
2. `==` walks values. `is` compares pointers. A second list with the same numbers is equal and not identical.
3. Rebinding stores a new pointer in one namespace slot. The other name, and the original object, stay as they were. `append` does not write either slot.
</details>


In [2]:
left = [1, 2]
right = left
right.append(3)
print(left)
print(left is right)
other = [1, 2, 3]
print(left == other)
print(left is other)


[1, 2, 3]
True
True
False


<details><summary>Check your answers</summary>

- `left` prints `[1, 2, 3]`. `right = left` did not copy. `append` changed the one list both names point at.
- `left is right` is `True`. One object, two names.
- `left == other` is `True`. The new list holds the same numbers.
- `left is other` is `False`. Equal values, two objects. `==` and `is` answered different questions.
</details>

Step that cell in [Python Tutor](https://pythontutor.com/visualize.html). You should see one list object and two arrows until `other` creates a second list.


## Why it works this way

An object sits in memory until nothing refers to it. A name is a slot in a namespace dictionary: the key is the string `"left"`, the value is a pointer to the object. `left = [1, 2]` creates a list object, then stores a pointer to it under `left`. `right = left` reads that pointer and stores the same pointer under `right`. No second list is allocated. `id` reads the identity of whatever the pointer currently reaches. `is` compares two pointers. That is why `id(left) == id(right)` and `left is right` are the same question, and why both stay true after `append`: `append` changes the object, not the pointers.

Rebinding is a different write. `right = [1, 2, 3]` allocates a new list and stores a new pointer under `right`. The slot `left` is untouched, so `left` still points at the original. Mutation and rebinding look similar in the source if you only watch the name on the left of `=`. They are not. Mutation calls a method on the object. Rebinding replaces the pointer in one slot.

The cost follows from that split. Writing a name is a constant-time pointer store, however large the object is. Copying a list is proportional to its length in time and in extra memory, because every element pointer is written into a new list. Sharing is the cheap default on purpose: a function can receive a list and update it without the caller paying for a copy, and two parts of a program can observe one buffer. The price is that a method call is visible through every name that still points there.

`==` is not a pointer compare. It calls the equality method, which for a list walks the elements. Two lists with the same numbers compare equal and still have different identities. Identity is the right question for "is this the same object I already hold", which is why `value is None` works: there is one `None` object. It is the wrong question for "do these numbers match".

CPython detail: small integers are sometimes one shared object, so `(1 + 1) is 2` can be true. That is a cache, not a rule. Other interpreters, and integers outside the cached range, do not promise it. The cell that prints it is marked volatile for that reason. Write `==` for numbers. The identity lesson in the pitfalls unit is the same warning from the other side.

The stable evidence is below: shared names compare `is`, a copy does not, and an append through one name is visible through the other only while they still share.


In [3]:
left = [1, 2]
right = left
print(left is right, id(left) == id(right))
right.append(3)
print(left, right, left is right)
right = [1, 2, 3]
print(left is right, left == right)
copied = list(left)
print(copied == left, copied is left)


True True
[1, 2, 3] [1, 2, 3] True
False True
True False


In [4]:
# CPython detail: small ints may be one cached object. Do not rely on this.
two = 2
print((1 + 1) is two)
big = 10**6
other = 10**6
print(big is other)


True
False


## When to use it

Use assignment when you want another name for the same object: an alias in a loop, a parameter, a shorter name for a long lookup. Use `list(samples)`, `samples.copy()`, or a slice when the next mutation must not be shared. Use `is` for `None` and for sentinels you created as a single object. Use `==` for values: numbers, strings, lists of readings.

Do not use `is` to compare integers or strings. A true result can be the cache, and a false result can be two equal objects. Do not assume a function argument was copied for you. The parameter name is another label on the caller's object, which is the sharing the functions unit describes.

A full deep copy of nested lists is a different tool, `copy.deepcopy`, and it belongs with the copying lesson. `list(samples)` copies one level. If the elements are themselves lists, those inner lists are still shared.


## Common mistake

Binding a second name and then mutating feels like editing a private copy. It is not. Both names print the appended list. Nothing raises. The bug is the belief that `=` copied.


In [5]:
readings = [1, 2]
backup = readings
backup.append(3)
print(readings)
print(backup)
print(readings is backup)


[1, 2, 3]
[1, 2, 3]
True


`readings` prints `[1, 2, 3]`. `backup` was never a second list. The fix builds one. After `list`, the append stays on `readings` and `backup` still prints `[1, 2]`.


In [6]:
readings = [1, 2]
backup = list(readings)
readings.append(3)
print(readings)
print(backup)
print(readings is backup)


[1, 2, 3]
[1, 2]
False


## Try it

Change `snapshot` so it returns the same list object the caller passed, and print `frozen is samples` after an append. Why does the frozen name now show 12, and which single call do you put back so the append is invisible to `frozen`?


## What this explains

- [Names](../../course/01-basics/01-names-lesson.ipynb): a name points at a value. This lesson is that pointer, and why a second name is not a second value.
- [Lists](../../course/04-collections/01-lists-lesson.ipynb): `append` changes the object. Every name that still points at it observes the change.
- [Identity](../../course/10-pitfalls/03-identity-lesson.ipynb): `is` versus `==`, and why `None` is a safe use of `is`.


## Recap

- Assignment stores a pointer. It does not copy the object.
- `is` compares identity. `==` compares values. Small-int caching is not a rule.
- Copy when a later mutation must stay private. Sharing is the cheap default.

Next: [Mutability and aliasing](../02-mutability/01-aliasing-lesson.ipynb).
